# Fig. 1：Ludlow16 浓度生成口径诊断

这个 Notebook 只诊断 Fig. 1，不修改 `haloconcentration.py`。目标是把三件事分开：

1. 从论文 PDF 的蓝色矢量路径恢复原 Fig. 1 浓度；
2. 比较 Ludlow16 Appendix C6 的出版公式与当前 $z>6$ 的 $\nu_0\propto(1+z)^{-1}$ 延拓；
3. 判断偏差是从 $z>6$ 才出现，还是在质量单位、质量吸积史或 $z<6$ 的公式阶段已经存在。

所有替代分支都只存在于本 Notebook。拟合原图得到的指数或逐点反求的 $\nu_0$ 只是诊断量，不会被当作 Ludlow16 的新公式。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pymupdf
from scipy.optimize import brentq, minimize_scalar

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'haloconcentration.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import haloconcentration as hc

PAPER_PATH = PROJECT_ROOT / 'Simulating Binary Primordial Black Hole Mergers in Dark Matter Halos.pdf'
PRESENT_DAY_MASS_MSUN = 1.0e12
REDSHIFT = np.linspace(0.0, 12.0, 100)
PIVOT_REDSHIFT = 6.0

print('项目目录：', PROJECT_ROOT)
print('论文 PDF：', PAPER_PATH)

## 1. 从原 PDF 恢复 Fig. 1 的蓝线和红线

下面读取的是 PDF 第 3 页内由 Matplotlib 保存的折线路径，不是复现 PNG。蓝色虚线和红色实线各有 99 段、100 个顶点。由于原图横轴为对数轴而数据网格含 $z=0$，第一个顶点不能可靠反算；CSV 中将这一行保留为 `NaN`，其余 99 个点可恢复。

In [ ]:
TAB_BLUE = np.array([0.1215686277, 0.4666666687, 0.7058823705])
TAB_RED = np.array([0.8392156959, 0.1529411823, 0.1568627506])

def drawing_vertices(drawing):
    line_items = [item for item in drawing['items'] if item[0] == 'l']
    points = [line_items[0][1]] + [item[2] for item in line_items]
    return np.array([[point.x, point.y] for point in points], dtype=float)

def find_fig1_curve(drawings, color):
    candidates = []
    for drawing in drawings:
        if drawing.get('color') is None:
            continue
        if not np.allclose(drawing['color'], color, atol=2.0e-5):
            continue
        if len(drawing['items']) != 99:
            continue
        if 40.0 <= drawing['rect'].x0 <= 70.0:
            candidates.append(drawing)
    if len(candidates) != 1:
        raise RuntimeError(f'Fig. 1 目标曲线候选数为 {len(candidates)}。')
    return candidates[0]

paper = pymupdf.open(PAPER_PATH)
page3_drawings = paper[2].get_drawings()
blue_xy = drawing_vertices(find_fig1_curve(page3_drawings, TAB_BLUE))
red_xy = drawing_vertices(find_fig1_curve(page3_drawings, TAB_RED))

# 横轴由 z=1、z=10 两个主刻度标定。
X_AT_Z1 = 164.937500
X_AT_Z10 = 245.524063
source_z_from_x = 10.0 ** ((blue_xy[:, 0] - X_AT_Z1) / (X_AT_Z10 - X_AT_Z1))
source_z_from_x[0] = np.nan

# 左轴是线性浓度轴：相邻整数刻度间隔为 23.945465 PDF 点。
Y_AT_C8 = 68.240417
CONCENTRATION_POINTS_PER_UNIT = 23.945465
source_concentration = 8.0 + (Y_AT_C8 - blue_xy[:, 1]) / CONCENTRATION_POINTS_PER_UNIT
source_concentration[0] = np.nan

# 右轴是对数质量轴：10^12 M_sun 主刻度及每十倍质量的间距。
Y_AT_MASS_1E12 = 68.744553
MASS_POINTS_PER_DECADE = 33.709030
source_mass_msun = 10.0 ** (
    12.0 + (Y_AT_MASS_1E12 - red_xy[:, 1]) / MASS_POINTS_PER_DECADE
)
source_mass_msun[0] = np.nan

# 另存原 PDF 的 Fig. 1 面板，便于肉眼确认数据来源。
fig1_panel = paper[2].get_pixmap(
    matrix=pymupdf.Matrix(2.5, 2.5),
    clip=pymupdf.Rect(35.0, 50.0, 305.0, 235.0),
)
fig1_panel.save(PROJECT_ROOT / 'fig1_pdf_source_panel.png')

print('蓝线顶点数：', len(blue_xy))
print('99 个可恢复点的最大红移坐标误差：',
      np.nanmax(np.abs(source_z_from_x - REDSHIFT)))

## 2. 把浓度公式写成“指定 $\nu_0$”的形式

Ludlow16 的平滑破幂律只通过比值 $\nu/\nu_0$ 使用 $\nu_0$。因此我们不改主体函数，而是在诊断函数中显式传入不同的 $\nu_0(z)$，其余 $c_0,\beta,\gamma_1,\gamma_2,\nu(M,z)$ 保持完全相同。

In [ ]:
def concentration_with_nu0(mass_msun, z, nu0):
    nu = hc.peak_height_ludlow16(mass_msun, z)
    ratio = nu / nu0
    beta = hc.ludlow_transition_beta(z)
    gamma1 = hc.ludlow_gamma1(z)
    gamma2 = hc.ludlow_gamma2(z)
    return (
        hc.ludlow_c0(z)
        * ratio ** (-gamma1)
        * (1.0 + ratio ** (1.0 / beta)) ** (-beta * (gamma2 - gamma1))
    )

published_nu0 = hc.ludlow_nu0_published(REDSHIFT)
nu0_at_pivot = float(hc.ludlow_nu0_published(PIVOT_REDSHIFT))

def continued_nu0(z, exponent):
    z = np.asarray(z, dtype=float)
    published = hc.ludlow_nu0_published(np.minimum(z, PIVOT_REDSHIFT))
    tail = nu0_at_pivot * ((1.0 + z) / (1.0 + PIVOT_REDSHIFT)) ** exponent
    return np.where(z <= PIVOT_REDSHIFT, published, tail)

# 出版多项式过零的位置。过零以后，非整数幂使浓度公式不再有实数值。
published_zero_redshift = brentq(
    lambda z: float(hc.ludlow_nu0_published(z)), 7.0, 9.0
)
print('Appendix C6 出版公式的 nu0=0 位于 z =', published_zero_redshift)
print('当前延拓在 z=6 的锚点 nu0(6) =', nu0_at_pivot)

## 3. 控制变量分支

分支含义：

- `current`：当前主体程序，$M(z)$、物理 $M_\odot\to h^{-1}M_\odot$ 换算、$z>6$ 指数 $-1$；
- `published_valid`：出版 $\nu_0$ 原式，只画 $\nu_0>0$ 的区间；
- `freeze_nu0`：在 $z>6$ 冻结 $\nu_0(6)$，只用来显示尾部对 $\nu_0$ 的敏感度；
- `best_tail_exponent`：仅用原图的 $z>6$ 点拟合延拓指数；
- `no_h_numeric_convention`：把质量数值直接代入 Appendix C 的 $m$，模拟常见软件以 $M_\odot/h$ 为输入时不再额外乘 $h$ 的口径；
- `source_mass`：只把红色矢量路径恢复出的 $M(z)$ 代入，判断质量史差异能否解释蓝线。

In [ ]:
current_mass_msun = hc.mass_accretion_history(
    PRESENT_DAY_MASS_MSUN, REDSHIFT, model='ludlow16'
)
current_nu0 = continued_nu0(REDSHIFT, exponent=-1.0)
current_concentration = concentration_with_nu0(
    current_mass_msun, REDSHIFT, current_nu0
)

published_concentration = np.full_like(REDSHIFT, np.nan)
published_positive = published_nu0 > 0.0
published_concentration[published_positive] = concentration_with_nu0(
    current_mass_msun[published_positive],
    REDSHIFT[published_positive],
    published_nu0[published_positive],
)

frozen_nu0 = np.where(
    REDSHIFT <= PIVOT_REDSHIFT, published_nu0, nu0_at_pivot
)
frozen_concentration = concentration_with_nu0(
    current_mass_msun, REDSHIFT, frozen_nu0
)

high_z = (REDSHIFT > PIVOT_REDSHIFT) & np.isfinite(source_concentration)
def tail_rms_dex(exponent, mass_for_model=current_mass_msun):
    trial = concentration_with_nu0(
        mass_for_model, REDSHIFT, continued_nu0(REDSHIFT, exponent)
    )
    return np.sqrt(np.mean(
        np.log10(trial[high_z] / source_concentration[high_z]) ** 2
    ))

tail_fit = minimize_scalar(
    tail_rms_dex, bounds=(-4.0, 2.0), method='bounded'
)
best_tail_exponent = float(tail_fit.x)
best_tail_nu0 = continued_nu0(REDSHIFT, best_tail_exponent)
best_tail_concentration = concentration_with_nu0(
    current_mass_msun, REDSHIFT, best_tail_nu0
)

# 主体接口接收物理 M_sun，内部会乘 h。除以 h 后再调用，可隔离“直接使用质量数值”口径。
h_ludlow = hc.LUDLOW_COSMOLOGY['h']
no_h_numeric_concentration = concentration_with_nu0(
    current_mass_msun / h_ludlow, REDSHIFT, current_nu0
)
source_mass_concentration = concentration_with_nu0(
    source_mass_msun, REDSHIFT, current_nu0
)

print('只拟合 z>6 得到的最优延拓指数：', best_tail_exponent)

## 4. 从原图逐点反求“有效 $\nu_0$”

这里固定原图红线给出的 $M(z)$，再求出使平滑破幂律恰好等于原图蓝线的 $\nu_0$。它吸收了质量单位、增长因子、绘图实现等所有未确认差异，因此只能称为**有效 $\nu_0$**，不能倒过来宣称作者真的使用了它。

In [ ]:
effective_nu0 = np.full_like(REDSHIFT, np.nan)
for index in range(1, REDSHIFT.size):
    z_value = REDSHIFT[index]
    mass_value = source_mass_msun[index]
    target_c = source_concentration[index]

    def squared_log_residual(log_nu0):
        trial_c = concentration_with_nu0(
            mass_value, z_value, np.exp(log_nu0)
        )
        return np.log(trial_c / target_c) ** 2

    result = minimize_scalar(
        squared_log_residual, bounds=(-10.0, 10.0), method='bounded'
    )
    effective_nu0[index] = np.exp(result.x)

print('代表性红移处：z, 原图 C, 当前 C, 当前 nu0, 有效 nu0')
for requested_z in [1.2, 3.6, 6.0, 7.3, 8.5, 9.7, 12.0]:
    i = int(np.argmin(np.abs(REDSHIFT - requested_z)))
    print(f'{REDSHIFT[i]:5.2f}  {source_concentration[i]:8.4f}  '
          f'{current_concentration[i]:8.4f}  {current_nu0[i]:8.4f}  '
          f'{effective_nu0[i]:8.4f}')

## 5. 误差表与数据导出

RMS 使用 $\log_{10}(C_\mathrm{model}/C_\mathrm{source})$，可避免高浓度端仅因绝对数值较大而占据过多权重。最大相对误差仍以百分比给出，便于直观判断。

In [ ]:
branches = {
    'current_p_minus_1': current_concentration,
    'published_positive_only': published_concentration,
    'freeze_nu0_after_z6': frozen_concentration,
    'best_tail_exponent': best_tail_concentration,
    'no_h_numeric_convention': no_h_numeric_concentration,
    'source_mass_with_current_nu0': source_mass_concentration,
}

def branch_metrics(values, mask):
    usable = mask & np.isfinite(values) & np.isfinite(source_concentration)
    log_residual = np.log10(values[usable] / source_concentration[usable])
    fractional = values[usable] / source_concentration[usable] - 1.0
    return (usable.sum(), np.sqrt(np.mean(log_residual**2)), np.max(np.abs(fractional)))

regions = {
    'all_z_gt_0': REDSHIFT > 0.0,
    'z_le_6': (REDSHIFT > 0.0) & (REDSHIFT <= PIVOT_REDSHIFT),
    'z_gt_6': REDSHIFT > PIVOT_REDSHIFT,
}
metric_rows = []
for branch_name, values in branches.items():
    for region_name, region_mask in regions.items():
        point_count, rms_dex, max_fraction = branch_metrics(values, region_mask)
        metric_rows.append((branch_name, region_name, point_count, rms_dex, max_fraction))

print(f"{'branch':31s} {'region':12s} {'N':>3s} {'RMS dex':>10s} {'max |dC/C|':>12s}")
for row in metric_rows:
    print(f'{row[0]:31s} {row[1]:12s} {row[2]:3d} {row[3]:10.5f} {row[4]:12.3%}')

source_table = np.column_stack([
    REDSHIFT, source_z_from_x, source_mass_msun, source_concentration
])
np.savetxt(
    PROJECT_ROOT / 'fig1_ludlow16_source_vector.csv',
    source_table, delimiter=',', comments='', fmt='%.12e',
    header='z_grid,z_recovered_from_pdf_x,source_mass_msun,source_concentration',
)

diagnostic_table = np.column_stack([
    REDSHIFT, current_mass_msun, source_mass_msun, source_concentration,
    current_concentration, published_concentration, frozen_concentration,
    best_tail_concentration, no_h_numeric_concentration,
    source_mass_concentration, published_nu0, current_nu0,
    best_tail_nu0, effective_nu0,
])
np.savetxt(
    PROJECT_ROOT / 'fig1_ludlow16_concentration_diagnostic.csv',
    diagnostic_table, delimiter=',', comments='', fmt='%.12e',
    header=(
        'z,current_mass_msun,source_mass_msun,source_concentration,'
        'current_concentration,published_concentration,freeze_nu0_concentration,'
        'best_tail_concentration,no_h_numeric_concentration,'
        'source_mass_current_nu0_concentration,published_nu0,current_nu0,'
        'best_tail_nu0,effective_nu0_from_source_mass'
    ),
)

with (PROJECT_ROOT / 'fig1_ludlow16_branch_metrics.csv').open('w', encoding='utf-8') as handle:
    handle.write('branch,region,point_count,rms_log10_dex,max_absolute_fractional_error\n')
    for row in metric_rows:
        handle.write(f'{row[0]},{row[1]},{row[2]},{row[3]:.12e},{row[4]:.12e}\n')

## 6. 可视化：曲线、残差、$\nu_0$ 和质量史

灰色竖线标出当前延拓起点 $z=6$；黑色点划线标出出版 $\nu_0$ 过零的位置。出版公式过零后的浓度没有实数意义，因此对应曲线在那里停止。

In [ ]:
positive_z = REDSHIFT > 0.0
figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.6))

ax = axes[0, 0]
ax.plot(REDSHIFT[positive_z], source_concentration[positive_z], 'k-', lw=2.4, label='PDF vector')
ax.plot(REDSHIFT[positive_z], current_concentration[positive_z], '--', lw=2.0, label='current: p=-1')
ax.plot(REDSHIFT[positive_z], best_tail_concentration[positive_z], ':', lw=2.0, label=f'best tail p={best_tail_exponent:.3f}')
ax.plot(REDSHIFT[positive_z], no_h_numeric_concentration[positive_z], '-.', lw=1.7, label='direct mass number')
ax.plot(REDSHIFT[positive_z], published_concentration[positive_z], color='tab:red', alpha=0.75, label='published nu0 where positive')
ax.set_xscale('log')
ax.set_xlabel('Redshift z')
ax.set_ylabel('Concentration C')
ax.set_title('Fig. 1 concentration')
ax.legend(fontsize=8)
ax.grid(alpha=0.22)

ax = axes[0, 1]
for label, values in [
    ('current: p=-1', current_concentration),
    (f'best tail p={best_tail_exponent:.3f}', best_tail_concentration),
    ('direct mass number', no_h_numeric_concentration),
    ('source M(z), current nu0', source_mass_concentration),
]:
    residual_percent = 100.0 * (values / source_concentration - 1.0)
    ax.plot(REDSHIFT[positive_z], residual_percent[positive_z], label=label)
ax.axhline(0.0, color='black', lw=0.8)
ax.axvline(PIVOT_REDSHIFT, color='0.5', ls='--', lw=1.0)
ax.set_xlabel('Redshift z')
ax.set_ylabel('100 x (model/source - 1) [%]')
ax.set_title('Residual to the PDF vector')
ax.legend(fontsize=8)
ax.grid(alpha=0.22)

ax = axes[1, 0]
ax.plot(REDSHIFT, published_nu0, color='tab:red', label='published Appendix C6')
ax.plot(REDSHIFT, current_nu0, '--', lw=2.0, label='current continuation p=-1')
ax.plot(REDSHIFT, best_tail_nu0, ':', lw=2.0, label=f'best tail p={best_tail_exponent:.3f}')
ax.plot(REDSHIFT[positive_z], effective_nu0[positive_z], color='black', lw=2.2, label='effective nu0 from PDF')
ax.axhline(0.0, color='black', lw=0.8)
ax.axvline(PIVOT_REDSHIFT, color='0.5', ls='--', lw=1.0)
ax.axvline(published_zero_redshift, color='black', ls='-.', lw=1.0)
ax.set_ylim(-1.0, 5.0)
ax.set_xlabel('Redshift z')
ax.set_ylabel('nu0')
ax.set_title('Published, continued and effective nu0')
ax.legend(fontsize=8)
ax.grid(alpha=0.22)

ax = axes[1, 1]
ax.plot(REDSHIFT[positive_z], source_mass_msun[positive_z], 'k-', lw=2.2, label='PDF red vector')
ax.plot(REDSHIFT[positive_z], current_mass_msun[positive_z], '--', lw=2.0, label='current M(z)')
ax.set_yscale('log')
ax.set_xlabel('Redshift z')
ax.set_ylabel('Halo mass [M_sun]')
ax.set_title('Mass history used by the blue curve')
ax.legend(fontsize=8)
ax.grid(alpha=0.22)

figure.tight_layout()
figure_path = PROJECT_ROOT / 'fig1_ludlow16_concentration_diagnosis.png'
figure.savefig(figure_path, dpi=220)
print('诊断图：', figure_path)
plt.show()

## 7. 如何读这份诊断

- 若偏差只由 $z>6$ 延拓造成，那么 `current` 在 $z\le6$ 应与 PDF 蓝线贴合，并且改变尾部指数应显著降低高红移残差。
- 若最优指数仍接近 $-1$，说明延拓的**斜率**不是主误差；接下来应检查质量数值是否被当作 $M_\odot/h$ 直接传入，以及原图是否还有未说明的整体归一化。
- `published_positive_only` 说明为什么不能简单删除延拓：Appendix C6 的多项式在约 $z=7.8$ 过零，此后原式不能直接用于非整数幂。
- `effective_nu0` 只显示原图要求的等效变化，不提供作者实现的唯一证据。只有取得作者源码或明确的软件版本后，才能把某个高红移口径认定为原图生成方式。